In [1]:
import os
os.environ["PYSPARK_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["PYSPARK_DRIVER_PYTHON"] = r"C:\Users\Ben\AppData\Local\Programs\Python\Python310\python.exe"
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"
os.environ["hadoop.home.dir"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import (
    col, row_number, rank, dense_rank,
    lag, lead, sum, avg, count
)
from pyspark.sql.window import Window

spark = SparkSession.builder.appName("Week2_Day3").master("local[*]").getOrCreate()

emp_data = [
    (1, "Alice",   "Engineering", 95000),
    (2, "Bob",     "Engineering", 90000),
    (3, "Charlie", "Engineering", 90000),
    (4, "Diana",   "Data",        85000),
    (5, "Eve",     "Data",        80000),
    (6, "Frank",   "Data",        80000),
    (7, "Grace",   "HR",          70000),
    (8, "Hank",    "HR",          65000),
]

emp_schema = StructType([
    StructField("emp_id", IntegerType(), False),
    StructField("name", StringType(), False),
    StructField("department", StringType(), False),
    StructField("salary", IntegerType(), False),
])

df_emp = spark.createDataFrame(emp_data, emp_schema)

# Monthly sales data (for running totals and lag/lead)
sales_data = [
    ("North", "Jan", 10000), ("North", "Feb", 15000),
    ("North", "Mar", 12000), ("North", "Apr", 18000),
    ("South", "Jan", 8000),  ("South", "Feb", 9000),
    ("South", "Mar", 11000), ("South", "Apr", 7000),
]

sales_schema = StructType([
    StructField("region", StringType(), False),
    StructField("month", StringType(), False),
    StructField("revenue", IntegerType(), False),
])

df_sales = spark.createDataFrame(sales_data, sales_schema)

In [3]:
# "Within each department, order employees by salary descending"
window_dept = Window.partitionBy("department").orderBy(col("salary").desc())

In [4]:
df_ranked = df_emp.withColumn(
    "row_num", 
    row_number().over(window_dept)
)

df_ranked.show()

+------+-------+-----------+------+-------+
|emp_id|   name| department|salary|row_num|
+------+-------+-----------+------+-------+
|     4|  Diana|       Data| 85000|      1|
|     5|    Eve|       Data| 80000|      2|
|     6|  Frank|       Data| 80000|      3|
|     1|  Alice|Engineering| 95000|      1|
|     2|    Bob|Engineering| 90000|      2|
|     3|Charlie|Engineering| 90000|      3|
|     7|  Grace|         HR| 70000|      1|
|     8|   Hank|         HR| 65000|      2|
+------+-------+-----------+------+-------+



In [5]:
df_ranked2 = df_emp.withColumn(
    "rank", 
    rank().over(window_dept)
)

df_ranked2.show()

+------+-------+-----------+------+----+
|emp_id|   name| department|salary|rank|
+------+-------+-----------+------+----+
|     4|  Diana|       Data| 85000|   1|
|     5|    Eve|       Data| 80000|   2|
|     6|  Frank|       Data| 80000|   2|
|     1|  Alice|Engineering| 95000|   1|
|     2|    Bob|Engineering| 90000|   2|
|     3|Charlie|Engineering| 90000|   2|
|     7|  Grace|         HR| 70000|   1|
|     8|   Hank|         HR| 65000|   2|
+------+-------+-----------+------+----+



In [6]:
window_monthly = Window.partitionBy("region").orderBy("month")

df_mon = df_sales.withColumn(
    "prev_month_revenue",
    lag("revenue",1).over(window_monthly)
)
df_mon.show()

+------+-----+-------+------------------+
|region|month|revenue|prev_month_revenue|
+------+-----+-------+------------------+
| North|  Apr|  18000|              NULL|
| North|  Feb|  15000|             18000|
| North|  Jan|  10000|             15000|
| North|  Mar|  12000|             10000|
| South|  Apr|   7000|              NULL|
| South|  Feb|   9000|              7000|
| South|  Jan|   8000|              9000|
| South|  Mar|  11000|              8000|
+------+-----+-------+------------------+



In [7]:
df_change = df_mon.withColumn(
    "change",
    col("revenue")-col("prev_month_revenue")
)

df_change.show()

+------+-----+-------+------------------+------+
|region|month|revenue|prev_month_revenue|change|
+------+-----+-------+------------------+------+
| North|  Apr|  18000|              NULL|  NULL|
| North|  Feb|  15000|             18000| -3000|
| North|  Jan|  10000|             15000| -5000|
| North|  Mar|  12000|             10000|  2000|
| South|  Apr|   7000|              NULL|  NULL|
| South|  Feb|   9000|              7000|  2000|
| South|  Jan|   8000|              9000| -1000|
| South|  Mar|  11000|              8000|  3000|
+------+-----+-------+------------------+------+



In [9]:
from pyspark.sql.functions import max
window_dept = Window.partitionBy("department").orderBy(col("salary").desc())

df_full = (
    df_emp
    .withColumn("rank", row_number().over(window_dept))
    .withColumn("dept_max_salary", max("salary").over(Window.partitionBy("department")))
    .withColumn("salary_gap_from_top", col("dept_max_salary") - col("salary"))
)

df_full.show()

+------+-------+-----------+------+----+---------------+-------------------+
|emp_id|   name| department|salary|rank|dept_max_salary|salary_gap_from_top|
+------+-------+-----------+------+----+---------------+-------------------+
|     4|  Diana|       Data| 85000|   1|          85000|                  0|
|     5|    Eve|       Data| 80000|   2|          85000|               5000|
|     6|  Frank|       Data| 80000|   3|          85000|               5000|
|     1|  Alice|Engineering| 95000|   1|          95000|                  0|
|     2|    Bob|Engineering| 90000|   2|          95000|               5000|
|     3|Charlie|Engineering| 90000|   3|          95000|               5000|
|     7|  Grace|         HR| 70000|   1|          70000|                  0|
|     8|   Hank|         HR| 65000|   2|          70000|               5000|
+------+-------+-----------+------+----+---------------+-------------------+



In [11]:
#EXERCISE

emp_data2 = [
    (1, "Alice", "Engineering", 9000),
    (2, "Bob", "Engineering", 8500),
    (3, "Charlie", "Engineering", 8500),   # tie
    (4, "David", "Engineering", 7000),

    (5, "Eve", "Sales", 8000),
    (6, "Frank", "Sales", 7800),
    (7, "Grace", "Sales", 7800),           # tie
    (8, "Heidi", "Sales", 6500),

    (9, "Ivan", "HR", 7200),
    (10, "Judy", "HR", 7200),              # tie
    (11, "Karl", "HR", 6800),
    (12, "Liam", "HR", 6000)
]

columns = ["emp_id", "employee_name", "department", "salary"]

df_emp2 = spark.createDataFrame(emp_data2, columns)

df_emp2.show()

+------+-------------+-----------+------+
|emp_id|employee_name| department|salary|
+------+-------------+-----------+------+
|     1|        Alice|Engineering|  9000|
|     2|          Bob|Engineering|  8500|
|     3|      Charlie|Engineering|  8500|
|     4|        David|Engineering|  7000|
|     5|          Eve|      Sales|  8000|
|     6|        Frank|      Sales|  7800|
|     7|        Grace|      Sales|  7800|
|     8|        Heidi|      Sales|  6500|
|     9|         Ivan|         HR|  7200|
|    10|         Judy|         HR|  7200|
|    11|         Karl|         HR|  6800|
|    12|         Liam|         HR|  6000|
+------+-------------+-----------+------+



In [12]:
sales_data2 = [
    ("North", "2024-01", 12000),
    ("North", "2024-02", 15000),
    ("North", "2024-03", 17000),
    ("North", "2024-04", 16000),

    ("South", "2024-01", 10000),
    ("South", "2024-02", 11000),
    ("South", "2024-03", 9000),
    ("South", "2024-04", 14000),

    ("East", "2024-01", 8000),
    ("East", "2024-02", 8500),
    ("East", "2024-03", 9200),
    ("East", "2024-04", 9800),

    ("West", "2024-01", 9500),
    ("West", "2024-02", 10000),
    ("West", "2024-03", 10500),
    ("West", "2024-04", 12000)
]

sales_cols = ["region", "month", "revenue"]

df_sales2 = spark.createDataFrame(sales_data2, sales_cols)

df_sales2.show()

+------+-------+-------+
|region|  month|revenue|
+------+-------+-------+
| North|2024-01|  12000|
| North|2024-02|  15000|
| North|2024-03|  17000|
| North|2024-04|  16000|
| South|2024-01|  10000|
| South|2024-02|  11000|
| South|2024-03|   9000|
| South|2024-04|  14000|
|  East|2024-01|   8000|
|  East|2024-02|   8500|
|  East|2024-03|   9200|
|  East|2024-04|   9800|
|  West|2024-01|   9500|
|  West|2024-02|  10000|
|  West|2024-03|  10500|
|  West|2024-04|  12000|
+------+-------+-------+



In [16]:
window_comp = Window.partitionBy("department").orderBy(col("salary").desc())

df_ranker= (
    df_emp2
    .withColumn("rank1",row_number().over(window_comp))
    .withColumn("rank2",rank().over(window_comp))
    .withColumn("rank3",dense_rank().over(window_comp))
)
df_ranker.show()

+------+-------------+-----------+------+-----+-----+-----+
|emp_id|employee_name| department|salary|rank1|rank2|rank3|
+------+-------------+-----------+------+-----+-----+-----+
|     1|        Alice|Engineering|  9000|    1|    1|    1|
|     2|          Bob|Engineering|  8500|    2|    2|    2|
|     3|      Charlie|Engineering|  8500|    3|    2|    2|
|     4|        David|Engineering|  7000|    4|    4|    3|
|     9|         Ivan|         HR|  7200|    1|    1|    1|
|    10|         Judy|         HR|  7200|    2|    1|    1|
|    11|         Karl|         HR|  6800|    3|    3|    2|
|    12|         Liam|         HR|  6000|    4|    4|    3|
|     5|          Eve|      Sales|  8000|    1|    1|    1|
|     6|        Frank|      Sales|  7800|    2|    2|    2|
|     7|        Grace|      Sales|  7800|    3|    2|    2|
|     8|        Heidi|      Sales|  6500|    4|    4|    3|
+------+-------------+-----------+------+-----+-----+-----+



In [20]:
df_top2 = df_emp2.withColumn("rank",row_number().over(window_comp))

okk= df_top2.filter(col("rank") <=2)
okk.show()

+------+-------------+-----------+------+----+
|emp_id|employee_name| department|salary|rank|
+------+-------------+-----------+------+----+
|     1|        Alice|Engineering|  9000|   1|
|     2|          Bob|Engineering|  8500|   2|
|     9|         Ivan|         HR|  7200|   1|
|    10|         Judy|         HR|  7200|   2|
|     5|          Eve|      Sales|  8000|   1|
|     6|        Frank|      Sales|  7800|   2|
+------+-------------+-----------+------+----+



In [33]:
window_running2= (
    Window
    .partitionBy("region")
    .orderBy(col("month"))
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

In [34]:
df_running = df_sales2.withColumn(
    "running_revenue",
    sum("revenue").over(window_running2)
)

df_running.show()

+------+-------+-------+---------------+
|region|  month|revenue|running_revenue|
+------+-------+-------+---------------+
|  East|2024-01|   8000|           8000|
|  East|2024-02|   8500|          16500|
|  East|2024-03|   9200|          25700|
|  East|2024-04|   9800|          35500|
| North|2024-01|  12000|          12000|
| North|2024-02|  15000|          27000|
| North|2024-03|  17000|          44000|
| North|2024-04|  16000|          60000|
| South|2024-01|  10000|          10000|
| South|2024-02|  11000|          21000|
| South|2024-03|   9000|          30000|
| South|2024-04|  14000|          44000|
|  West|2024-01|   9500|           9500|
|  West|2024-02|  10000|          19500|
|  West|2024-03|  10500|          30000|
|  West|2024-04|  12000|          42000|
+------+-------+-------+---------------+



In [39]:

window_monthly = Window.partitionBy("region").orderBy("month")
df_mo=df_sales2.withColumn(
    "prev_revenue",
    lag("revenue",1).over(window_monthly)
)

df_mo.show()

diff = df_mo.withColumn(
    "rev_change",
    col("revenue")-col("prev_revenue")
)

diff.show()

+------+-------+-------+------------+
|region|  month|revenue|prev_revenue|
+------+-------+-------+------------+
|  East|2024-01|   8000|        NULL|
|  East|2024-02|   8500|        8000|
|  East|2024-03|   9200|        8500|
|  East|2024-04|   9800|        9200|
| North|2024-01|  12000|        NULL|
| North|2024-02|  15000|       12000|
| North|2024-03|  17000|       15000|
| North|2024-04|  16000|       17000|
| South|2024-01|  10000|        NULL|
| South|2024-02|  11000|       10000|
| South|2024-03|   9000|       11000|
| South|2024-04|  14000|        9000|
|  West|2024-01|   9500|        NULL|
|  West|2024-02|  10000|        9500|
|  West|2024-03|  10500|       10000|
|  West|2024-04|  12000|       10500|
+------+-------+-------+------------+

+------+-------+-------+------------+----------+
|region|  month|revenue|prev_revenue|rev_change|
+------+-------+-------+------------+----------+
|  East|2024-01|   8000|        NULL|      NULL|
|  East|2024-02|   8500|        8000|      